# Opening Range Breakout (ORB) Strategy - Backtest

Based on the strategy described in [@bennnytrades](https://www.instagram.com/bennnytrades/) reel.

**Strategy Rules:**
1. Define the Opening Range: 9:30 AM - 9:45 AM Eastern (first 15 minutes)
2. Breakout: Price closes above/below the range high/low
3. Regime Filters: ATR (volatility) and Relative Volume
4. Entry: On breakout candle close
5. Stop Loss: Opposite side of the opening range
6. Take Profit: Based on risk:reward ratio

**Note:** Since free intraday data is limited, we'll use two approaches:
- Daily simulation using the first candle as a proxy for ORB range
- Intraday with yfinance (limited to ~60 days for minute data)

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

## 1. Download Intraday Data (5-min candles)

Using SPY (S&P 500 ETF) as a proxy for ES futures. yfinance provides ~60 days of 5-min data for free.

In [ ]:
# Download 5-minute data for SPY (proxy for ES futures)
ticker = "SPY"
data = yf.download(ticker, period="60d", interval="5m", progress=False)
data.index = data.index.tz_convert("US/Eastern")

# Flatten multi-level columns if present
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

print(f"Downloaded {len(data)} bars from {data.index[0]} to {data.index[-1]}")
data.head()

## 2. Identify Opening Range (9:30 - 9:45 AM)

The ORB is the high and low of the first 15 minutes of trading (first 3 five-minute candles).

In [ ]:
def identify_opening_range(data):
    """
    For each trading day, identify the opening range (9:30-9:45 AM ET).
    Returns a DataFrame with one row per day: date, orb_high, orb_low, orb_volume.
    """
    data = data.copy()
    data['date'] = data.index.date
    data['time'] = data.index.time
    
    # Opening range: 9:30 to 9:45 (first 3 five-min candles)
    from datetime import time as dt_time
    orb_mask = (data['time'] >= dt_time(9, 30)) & (data['time'] < dt_time(9, 45))
    orb_bars = data[orb_mask]
    
    orb_stats = orb_bars.groupby('date').agg(
        orb_high=('High', 'max'),
        orb_low=('Low', 'min'),
        orb_volume=('Volume', 'sum')
    ).reset_index()
    
    return orb_stats

orb_levels = identify_opening_range(data)
print(f"Identified opening ranges for {len(orb_levels)} trading days")
orb_levels.head(10)

## 3. Calculate Regime Filters

- **ATR (Average True Range):** Measures volatility. We filter for days with sufficient volatility.
- **Relative Volume:** Compare ORB volume to the average. High relative volume confirms breakout strength.

In [ ]:
def calculate_regime_filters(data, orb_levels, atr_period=14, rvol_period=10):
    """
    Add ATR and Relative Volume filters to the ORB levels.
    """
    from datetime import time as dt_time
    
    data = data.copy()
    data['date'] = data.index.date
    
    # Get daily OHLC for ATR calculation
    daily = data.groupby('date').agg(
        daily_high=('High', 'max'),
        daily_low=('Low', 'min'),
        daily_close=('Close', 'last'),
        daily_open=('Open', 'first'),
        daily_volume=('Volume', 'sum')
    ).reset_index()
    
    # Calculate ATR
    daily['prev_close'] = daily['daily_close'].shift(1)
    daily['tr'] = np.maximum(
        daily['daily_high'] - daily['daily_low'],
        np.maximum(
            abs(daily['daily_high'] - daily['prev_close']),
            abs(daily['daily_low'] - daily['prev_close'])
        )
    )
    daily['atr'] = daily['tr'].rolling(window=atr_period).mean()
    
    # Merge with ORB levels
    orb = orb_levels.merge(daily[['date', 'atr', 'daily_volume', 'daily_close']], on='date', how='left')
    
    # Relative Volume: ORB volume vs average ORB volume over past N days
    orb['avg_orb_volume'] = orb['orb_volume'].rolling(window=rvol_period).mean()
    orb['relative_volume'] = orb['orb_volume'] / orb['avg_orb_volume']
    
    # ORB range as percentage of price
    orb['orb_range'] = orb['orb_high'] - orb['orb_low']
    orb['orb_range_pct'] = orb['orb_range'] / orb['daily_close'] * 100
    
    return orb

orb_with_filters = calculate_regime_filters(data, orb_levels)
print(f"\nFilter stats:")
print(orb_with_filters[['atr', 'relative_volume', 'orb_range_pct']].describe())
orb_with_filters.tail()

## 4. Run ORB Backtest

**Entry Rules:**
- Long: Price closes above ORB high after 9:45 AM
- Short: Price closes below ORB low after 9:45 AM

**Filters:**
- Relative Volume > threshold (confirms participation)
- ATR within acceptable range (not too dead, not too wild)

**Exit Rules:**
- Stop Loss: Opposite side of ORB range
- Take Profit: Risk:Reward ratio (default 1.5:1)
- Time stop: Close by 3:55 PM if neither hit

In [ ]:
def backtest_orb(
    data, 
    orb_with_filters,
    rvol_threshold=0.8,
    rr_ratio=1.5,
    max_trades_per_day=1,
    use_atr_filter=True,
    atr_min_pctile=20,
    atr_max_pctile=80
):
    """
    Backtest the ORB strategy on intraday data.
    
    Args:
        rvol_threshold: Minimum relative volume to take a trade
        rr_ratio: Risk:Reward ratio for take profit
        max_trades_per_day: Max trades allowed per day
        use_atr_filter: Whether to filter by ATR percentile
        atr_min_pctile: Min ATR percentile to trade
        atr_max_pctile: Max ATR percentile to trade
    """
    from datetime import time as dt_time
    
    data = data.copy()
    data['date'] = data.index.date
    data['time'] = data.index.time
    
    # Calculate ATR percentile thresholds
    atr_values = orb_with_filters['atr'].dropna()
    atr_low = atr_values.quantile(atr_min_pctile / 100)
    atr_high = atr_values.quantile(atr_max_pctile / 100)
    
    trades = []
    
    for _, orb_row in orb_with_filters.iterrows():
        date = orb_row['date']
        orb_high = orb_row['orb_high']
        orb_low = orb_row['orb_low']
        rvol = orb_row['relative_volume']
        atr = orb_row['atr']
        
        # Skip if filters not met
        if pd.isna(rvol) or pd.isna(atr):
            continue
        if rvol < rvol_threshold:
            continue
        if use_atr_filter and (atr < atr_low or atr > atr_high):
            continue
        
        # Get bars after the opening range for this day
        day_bars = data[(data['date'] == date) & (data['time'] >= dt_time(9, 45)) & (data['time'] <= dt_time(15, 55))]
        
        if len(day_bars) == 0:
            continue
        
        orb_range = orb_high - orb_low
        if orb_range <= 0:
            continue
        
        trade_taken = False
        
        for idx, bar in day_bars.iterrows():
            if trade_taken:
                break
            
            # LONG breakout
            if bar['Close'] > orb_high:
                entry = bar['Close']
                stop_loss = orb_low
                risk = entry - stop_loss
                take_profit = entry + (risk * rr_ratio)
                
                # Simulate trade outcome using remaining bars
                remaining = day_bars.loc[idx:]
                result = simulate_trade(remaining, entry, stop_loss, take_profit, 'long')
                result['date'] = date
                result['direction'] = 'long'
                result['entry'] = entry
                result['stop_loss'] = stop_loss
                result['take_profit'] = take_profit
                result['orb_range'] = orb_range
                result['rvol'] = rvol
                trades.append(result)
                trade_taken = True
            
            # SHORT breakout
            elif bar['Close'] < orb_low:
                entry = bar['Close']
                stop_loss = orb_high
                risk = stop_loss - entry
                take_profit = entry - (risk * rr_ratio)
                
                remaining = day_bars.loc[idx:]
                result = simulate_trade(remaining, entry, stop_loss, take_profit, 'short')
                result['date'] = date
                result['direction'] = 'short'
                result['entry'] = entry
                result['stop_loss'] = stop_loss
                result['take_profit'] = take_profit
                result['orb_range'] = orb_range
                result['rvol'] = rvol
                trades.append(result)
                trade_taken = True
    
    return pd.DataFrame(trades)


def simulate_trade(bars, entry, stop_loss, take_profit, direction):
    """
    Simulate a trade through subsequent bars.
    Returns dict with exit_price, exit_reason, pnl.
    """
    for idx, bar in bars.iterrows():
        if direction == 'long':
            # Check stop loss hit
            if bar['Low'] <= stop_loss:
                return {'exit_price': stop_loss, 'exit_reason': 'stop_loss', 'pnl': stop_loss - entry}
            # Check take profit hit
            if bar['High'] >= take_profit:
                return {'exit_price': take_profit, 'exit_reason': 'take_profit', 'pnl': take_profit - entry}
        else:  # short
            if bar['High'] >= stop_loss:
                return {'exit_price': stop_loss, 'exit_reason': 'stop_loss', 'pnl': entry - stop_loss}
            if bar['Low'] <= take_profit:
                return {'exit_price': take_profit, 'exit_reason': 'take_profit', 'pnl': entry - take_profit}
    
    # Time stop - close at last bar
    last_close = bars.iloc[-1]['Close']
    if direction == 'long':
        pnl = last_close - entry
    else:
        pnl = entry - last_close
    
    return {'exit_price': last_close, 'exit_reason': 'time_stop', 'pnl': pnl}


print("Backtest functions defined.")

In [ ]:
# Run the backtest
trades = backtest_orb(
    data,
    orb_with_filters,
    rvol_threshold=0.8,
    rr_ratio=1.5,
    use_atr_filter=True,
    atr_min_pctile=20,
    atr_max_pctile=80
)

print(f"Total trades: {len(trades)}")
if len(trades) > 0:
    print(f"\nTrade breakdown:")
    print(trades['exit_reason'].value_counts())
    print(f"\nDirection breakdown:")
    print(trades['direction'].value_counts())

## 5. Strategy Performance Report

In [ ]:
def performance_report(trades):
    """Generate a performance report from trade results."""
    if len(trades) == 0:
        print("No trades to analyze.")
        return
    
    total_trades = len(trades)
    winners = trades[trades['pnl'] > 0]
    losers = trades[trades['pnl'] <= 0]
    
    win_rate = len(winners) / total_trades * 100
    avg_win = winners['pnl'].mean() if len(winners) > 0 else 0
    avg_loss = losers['pnl'].mean() if len(losers) > 0 else 0
    
    total_pnl = trades['pnl'].sum()
    avg_pnl = trades['pnl'].mean()
    
    # Risk:Reward achieved
    rr_achieved = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')
    
    # Profit factor
    gross_profit = winners['pnl'].sum() if len(winners) > 0 else 0
    gross_loss = abs(losers['pnl'].sum()) if len(losers) > 0 else 0
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Max drawdown on cumulative PnL
    cum_pnl = trades['pnl'].cumsum()
    peak = cum_pnl.expanding().max()
    drawdown = cum_pnl - peak
    max_dd = drawdown.min()
    
    # Consecutive wins/losses
    streak = (trades['pnl'] > 0).astype(int)
    
    print("="*60)
    print("           ORB STRATEGY PERFORMANCE REPORT")
    print("="*60)
    print(f"\n{'Total Trades:':<25} {total_trades}")
    print(f"{'Winners:':<25} {len(winners)}")
    print(f"{'Losers:':<25} {len(losers)}")
    print(f"{'Win Rate:':<25} {win_rate:.1f}%")
    print(f"\n{'Total PnL (points):':<25} {total_pnl:.2f}")
    print(f"{'Avg PnL per trade:':<25} {avg_pnl:.2f}")
    print(f"{'Avg Winner:':<25} {avg_win:.2f}")
    print(f"{'Avg Loser:':<25} {avg_loss:.2f}")
    print(f"{'Risk:Reward Achieved:':<25} {rr_achieved:.2f}")
    print(f"{'Profit Factor:':<25} {profit_factor:.2f}")
    print(f"{'Max Drawdown (points):':<25} {max_dd:.2f}")
    print(f"\n{'Exit Breakdown:'}")
    for reason, count in trades['exit_reason'].value_counts().items():
        print(f"  {reason:<20} {count} ({count/total_trades*100:.1f}%)")
    print("="*60)
    
    return {
        'total_trades': total_trades,
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'rr_achieved': rr_achieved,
        'total_pnl': total_pnl,
        'max_drawdown': max_dd
    }

stats = performance_report(trades)

## 6. Equity Curve

In [ ]:
import matplotlib.pyplot as plt

if len(trades) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Equity curve
    cum_pnl = trades['pnl'].cumsum()
    axes[0, 0].plot(cum_pnl.values, linewidth=2, color='green')
    axes[0, 0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[0, 0].set_title('Equity Curve (Cumulative PnL)', fontsize=12)
    axes[0, 0].set_xlabel('Trade #')
    axes[0, 0].set_ylabel('Points')
    axes[0, 0].grid(True, alpha=0.3)
    
    # PnL distribution
    axes[0, 1].hist(trades['pnl'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_title('PnL Distribution', fontsize=12)
    axes[0, 1].set_xlabel('Points')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Win/Loss by direction
    direction_stats = trades.groupby('direction')['pnl'].agg(['mean', 'sum', 'count'])
    direction_stats['sum'].plot(kind='bar', ax=axes[1, 0], color=['coral', 'mediumseagreen'])
    axes[1, 0].set_title('Total PnL by Direction', fontsize=12)
    axes[1, 0].set_ylabel('Points')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=0)
    
    # Drawdown
    peak = cum_pnl.expanding().max()
    drawdown = cum_pnl - peak
    axes[1, 1].fill_between(range(len(drawdown)), drawdown.values, 0, color='red', alpha=0.3)
    axes[1, 1].plot(drawdown.values, color='red', linewidth=1)
    axes[1, 1].set_title('Drawdown', fontsize=12)
    axes[1, 1].set_xlabel('Trade #')
    axes[1, 1].set_ylabel('Points')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('orb_backtest_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nChart saved to orb_backtest_results.png")
else:
    print("No trades to plot.")

## 7. Parameter Optimization

Test different combinations of:
- Risk:Reward ratios
- Relative Volume thresholds
- ATR filter ranges

In [ ]:
def optimize_parameters(data, orb_with_filters):
    """Test multiple parameter combinations and rank by profit factor."""
    
    rr_ratios = [1.0, 1.5, 2.0, 2.5, 3.0]
    rvol_thresholds = [0.5, 0.8, 1.0, 1.2, 1.5]
    
    results = []
    
    for rr in rr_ratios:
        for rvol in rvol_thresholds:
            trades = backtest_orb(
                data,
                orb_with_filters,
                rvol_threshold=rvol,
                rr_ratio=rr,
                use_atr_filter=True
            )
            
            if len(trades) < 5:
                continue
            
            winners = trades[trades['pnl'] > 0]
            losers = trades[trades['pnl'] <= 0]
            win_rate = len(winners) / len(trades) * 100
            total_pnl = trades['pnl'].sum()
            gross_profit = winners['pnl'].sum() if len(winners) > 0 else 0
            gross_loss = abs(losers['pnl'].sum()) if len(losers) > 0 else 0
            pf = gross_profit / gross_loss if gross_loss > 0 else float('inf')
            
            results.append({
                'rr_ratio': rr,
                'rvol_threshold': rvol,
                'trades': len(trades),
                'win_rate': round(win_rate, 1),
                'total_pnl': round(total_pnl, 2),
                'profit_factor': round(pf, 2),
                'avg_pnl': round(trades['pnl'].mean(), 3)
            })
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('profit_factor', ascending=False)
    return results_df

print("Running parameter optimization...")
opt_results = optimize_parameters(data, orb_with_filters)
print(f"\nTop 10 parameter combinations (by Profit Factor):")
print("="*80)
opt_results.head(10)

## 8. Compare vs Buy & Hold

In [ ]:
if len(trades) > 0:
    # ORB strategy cumulative return
    orb_cum_pnl = trades['pnl'].cumsum()
    
    # Buy & Hold return over same period
    first_date = trades.iloc[0]['date']
    last_date = trades.iloc[-1]['date']
    
    bh_data = data[(data.index.date >= first_date) & (data.index.date <= last_date)]
    bh_start = bh_data.iloc[0]['Close']
    bh_end = bh_data.iloc[-1]['Close']
    bh_return = bh_end - bh_start
    
    print(f"Period: {first_date} to {last_date}")
    print(f"\n{'Strategy PnL (points):':<30} {orb_cum_pnl.iloc[-1]:.2f}")
    print(f"{'Buy & Hold Return (points):':<30} {bh_return:.2f}")
    print(f"{'Strategy vs B&H:':<30} {'OUTPERFORMED' if orb_cum_pnl.iloc[-1] > bh_return else 'UNDERPERFORMED'}")
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Normalize both to start at 0
    ax.plot(range(len(orb_cum_pnl)), orb_cum_pnl.values, label='ORB Strategy', linewidth=2, color='green')
    
    # Create buy & hold line scaled to same x-axis
    bh_line = np.linspace(0, bh_return, len(orb_cum_pnl))
    ax.plot(range(len(orb_cum_pnl)), bh_line, label='Buy & Hold', linewidth=2, color='blue', linestyle='--')
    
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('ORB Strategy vs Buy & Hold', fontsize=14)
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Points')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('orb_vs_buyhold.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No trades to compare.")

## 9. Quick Test with Different Ticker

Try QQQ (Nasdaq) or other liquid instruments.

In [ ]:
# Change ticker and re-run
# ticker = "QQQ"  # Nasdaq
# ticker = "IWM"  # Russell 2000
# ticker = "DIA"  # Dow Jones

# Uncomment below to run on a different ticker:
# new_data = yf.download(ticker, period="60d", interval="5m", progress=False)
# new_data.index = new_data.index.tz_convert("US/Eastern")
# if isinstance(new_data.columns, pd.MultiIndex):
#     new_data.columns = new_data.columns.get_level_values(0)
# new_orb = identify_opening_range(new_data)
# new_orb_filtered = calculate_regime_filters(new_data, new_orb)
# new_trades = backtest_orb(new_data, new_orb_filtered)
# performance_report(new_trades)